In [1]:
import sys
print(sys.executable)

d:\ML project\scania-predictive-maintenance\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.preprocessing import load_aps_csv, split_development_data

# Load the training CSV.
X, y = load_aps_csv(
    project_root / "data" / "aps_failure_training_set.csv"
)

# Create the reproducible 80/20 split.
X_dev, X_holdout, y_dev, y_holdout = split_development_data(X, y)

# Check the sizes, class counts, and separation.
assert X_dev.shape == (48000, 170)
assert X_holdout.shape == (12000, 170)
assert y_dev.value_counts().to_dict() == {0: 47200, 1: 800}
assert y_holdout.value_counts().to_dict() == {0: 11800, 1: 200}
assert X_dev.index.intersection(X_holdout.index).empty

print(f"Development: {len(y_dev):,} rows, {y_dev.sum():,} APS failures")
print(f"Internal holdout: {len(y_holdout):,} rows, {y_holdout.sum():,} APS failures")

Development: 48,000 rows, 800 APS failures
Internal holdout: 12,000 rows, 200 APS failures


In [3]:
import pandas as pd

missing_summary = pd.DataFrame({
    "Missing count": X_dev.isna().sum(),
    "Missing (%)": X_dev.isna().mean() * 100,
    "Non-APS missing (%)": (
        X_dev.loc[y_dev == 0].isna().mean() * 100
    ),
    "APS failure missing (%)": (
        X_dev.loc[y_dev == 1].isna().mean() * 100
    ),
})

missing_summary = missing_summary.sort_values(
    "Missing (%)",
    ascending=False,
)

display(missing_summary.head(10).round(2))

,Missing count,Missing (%),Non-APS missing (%),APS failure missing (%)
br_000,39438,82.16,83.40,9.25
bq_000,39025,81.30,82.53,8.88
bp_000,38244,79.68,80.88,8.38
bo_000,37146,77.39,78.57,7.88
ab_000,37031,77.15,77.15,76.75
cr_000,37031,77.15,77.15,76.75
bn_000,35301,73.54,74.66,7.62
bm_000,31746,66.14,67.13,7.38
bl_000,21845,45.51,46.16,7.00
bk_000,18445,38.43,38.97,6.62


In [4]:
# Count missing sensor readings in each development record.
row_missing = X_dev.isna().sum(axis=1)

row_report = pd.DataFrame({
    "Class": y_dev.map({0: "Non-APS", 1: "APS failure"}),
    "Missing readings": row_missing,
})

# Summarize missing readings for each class.
display(
    row_report.groupby("Class")["Missing readings"]
    .agg(["count", "mean", "median", "max"])
    .round(2)
)

# Inspect records with more than 100 missing sensor readings.
display(
    pd.crosstab(
        row_report["Class"],
        row_missing > 100,
        colnames=["More than 100 missing readings"],
    )
)

,count,mean,median,max
Class,,,,
APS failure,800,29.02,20.5,168
Non-APS,47200,13.90,8.0,168


More than 100 missing readings,False,True
Class,,
APS failure,798,2
Non-APS,46976,224


In [5]:
# Count distinct observed values, excluding missing values.
unique_counts = X_dev.nunique(dropna=True)

feature_check = pd.DataFrame({
    "Unique observed values": unique_counts,
    "Missing (%)": X_dev.isna().mean() * 100,
})

# Show completely empty columns and columns with one observed value.
display(
    feature_check.loc[
        feature_check["Unique observed values"] <= 1
    ].round(2)
)

print("Completely empty columns:", int((unique_counts == 0).sum()))
print("Constant observed-value columns:", int((unique_counts == 1).sum()))

,Unique observed values,Missing (%)
cd_000,1,1.14


Completely empty columns: 0
Constant observed-value columns: 1


In [6]:
import numpy as np
from src.preprocessing import make_preprocessor

# Learn preprocessing rules using development data only.
preprocessor = make_preprocessor()

X_dev_clean = preprocessor.fit_transform(X_dev)

# Apply the same rules to the internal holdout.
X_holdout_clean = preprocessor.transform(X_holdout)

# Identify the retained and removed columns.
kept_columns = preprocessor.get_feature_names_out()
removed_columns = [
    column for column in X_dev.columns
    if column not in kept_columns
]

# Check that rows are preserved and all values are finite.
assert X_dev_clean.shape == (48000, 169)
assert X_holdout_clean.shape == (12000, 169)
assert np.isfinite(X_dev_clean).all()
assert np.isfinite(X_holdout_clean).all()
assert removed_columns == ["cd_000"]

print("Development shape:", X_dev_clean.shape)
print("Internal holdout shape:", X_holdout_clean.shape)
print("Removed columns:", removed_columns)
print("All preprocessing checks passed.")

Development shape: (48000, 169)
Internal holdout shape: (12000, 169)
Removed columns: ['cd_000']
All preprocessing checks passed.


In [7]:
# Check all remaining preprocessing options.
for scale, indicators in [(True, False), (False, True), (True, True)]:
    processor = make_preprocessor(
        scale=scale,
        add_indicators=indicators,
    )

    dev_result = processor.fit_transform(X_dev)
    holdout_result = processor.transform(X_holdout)
    feature_names = processor.get_feature_names_out()

    # Both datasets must have matching columns and valid numbers.
    assert dev_result.shape[0] == len(X_dev)
    assert holdout_result.shape[0] == len(X_holdout)
    assert dev_result.shape[1] == holdout_result.shape[1]
    assert np.isfinite(dev_result).all()
    assert np.isfinite(holdout_result).all()
    assert "cd_000" not in feature_names

    if indicators:
        # Preserve information about missing cd_000 readings.
        flag_name = "missingindicator_cd_000"
        assert flag_name in feature_names

        if not scale:
            flag_index = list(feature_names).index(flag_name)
            np.testing.assert_array_equal(
                dev_result[:, flag_index],
                X_dev["cd_000"].isna().astype(int).to_numpy(),
            )

    if scale:
        # Standardization is checked on the fitted development data.
        assert np.allclose(dev_result.mean(axis=0), 0, atol=1e-6)
        assert np.allclose(dev_result.std(axis=0), 1, atol=1e-6)

    print(
        f"PASS: scale={scale}, indicators={indicators}, "
        f"features={dev_result.shape[1]}"
    )

print("All preprocessing options passed.")

PASS: scale=True, indicators=False, features=169
PASS: scale=False, indicators=True, features=338
PASS: scale=True, indicators=True, features=338
All preprocessing options passed.
